# Pipeline Accuracy & Scale Tests

> Direct pipeline invocation — no HTTP server needed.  
> Full LangGraph pipeline: **supervisor → math_agent → reviewer → formatter → END**

| # | Test | Topic | Items | Game Types | Kiểm tra |
|---|------|-------|-------|------------|----------|
| 1 | Accuracy 10Q | Đạo hàm và ứng dụng | 10 | quiz | Computation traces, 4 options, correct answer |
| 2 | Vietnamese | Phương trình bậc hai | 10 | quiz + flashcard | All content in Vietnamese |
| 3 | Diversity 20Q | Hàm số và đồ thị | 20 | quiz | Unique questions, sub-topic distribution |
| 4 | Scale 30Q | Tích phân và ứng dụng | 30 | quiz | Handles Kahoot-scale, time per item |
| 5 | All Game Types | Chuyển động thẳng đều | 20 | quiz+flashcard+fill_blank | All 3 formats work |

## Setup & Imports

In [1]:
import sys, time, re, json
from pathlib import Path
from IPython.display import display, Markdown, HTML

project_root = str(Path.cwd().parent.parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.config import get_settings
from src.api.schemas import (
    GenerationRequest, GameType, DifficultyLevel, DocScope,
    GameContentResponse, QuizQuestion, Flashcard, FillBlankQuestion,
)
from src.graph.builder import compile_graph
from src.graph.state import MAX_REVIEW_ITERATIONS

settings = get_settings()

display(Markdown(f"""
### Configuration
| Setting | Value |
|---------|-------|
| GCP Project | `{settings.gcp_project_id}` |
| Generation Model | `{settings.generation_model}` @ `{settings.generation_model_location or settings.gcp_location}` |
| Review Model | `{settings.review_model}` @ `{settings.review_model_location}` |
| Max Review Iterations | `{MAX_REVIEW_ITERATIONS}` |
"""))
print('Setup OK ✓')

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



### Configuration
| Setting | Value |
|---------|-------|
| GCP Project | `green-mercury-485016-n1` |
| Generation Model | `gemini-2.5-flash` @ `asia-southeast1` |
| Review Model | `gemini-3.1-flash-lite-preview` @ `global` |
| Max Review Iterations | `3` |


Setup OK ✓


## Helper Functions

In [2]:
# ── Request builder ──
def make_request(
    topic: str,
    num_questions: int = 10,
    game_types: list[str] | None = None,
    difficulty: str = 'comprehension',
    doc_scope: str = 'system',
) -> GenerationRequest:
    gts = [GameType(g) for g in (game_types or ['quiz'])]
    return GenerationRequest(
        user_id='accuracy_test',
        topic=topic,
        game_types=gts,
        num_questions=num_questions,
        difficulty=DifficultyLevel(difficulty),
        doc_scope=DocScope(doc_scope),
    )

# ── Pipeline runner ──
async def run_pipeline(request: GenerationRequest) -> dict:
    app = compile_graph()
    initial_state = {
        'request': request,
        'doc_scope': request.doc_scope.value,
        'iteration_count': 0,
        'rejected_items': [],
        'errors': [],
    }
    t0 = time.time()
    result = await app.ainvoke(initial_state)
    elapsed = time.time() - t0
    return {**result, '_elapsed': elapsed}

# ── Vietnamese detection ──
VN_CHARS = re.compile(r'[àáảãạăắằẳẵặâấầẩẫậèéẻẽẹêếềểễệìíỉĩịòóỏõọôốồổỗộơớờởỡợùúủũụưứừửữựỳýỷỹỵđ]', re.IGNORECASE)
def is_vietnamese(text: str) -> bool:
    return bool(VN_CHARS.search(text))

# ── Rich display helpers ──
def show_summary_table(output: GameContentResponse, elapsed: float, iterations: int) -> dict:
    c = output.content
    m = output.metadata
    qn, fn, bn = len(c.quiz), len(c.flashcard), len(c.fill_blank)
    total = qn + fn + bn
    pr = m.total_passed_review / m.total_generated * 100 if m.total_generated > 0 else 0

    display(Markdown(f"""
| Metric | Value |
|--------|-------|
| Total items | **{total}** |
| Quiz / Flashcard / Fill-blank | {qn} / {fn} / {bn} |
| Generated → Passed review | {m.total_generated} → {m.total_passed_review} ({pr:.0f}%) |
| Rejected | {m.total_rejected} |
| Iterations | {iterations} |
| Time | {elapsed:.1f}s ({elapsed/total:.1f}s per item) |
"""))

    return {'quiz': qn, 'fc': fn, 'fb': bn, 'total': total, 'elapsed': elapsed,
            'generated': m.total_generated, 'passed': m.total_passed_review, 'rejected': m.total_rejected}

def show_quiz_card(q: QuizQuestion, idx: int):
    correct_opt = next((o for o in q.options if o.is_correct), None)
    options_md = '\n'.join(f'  - {"✅" if o.is_correct else "❌"} {o.text}' for o in q.options)
    trace_md = ''
    if q.computation_trace:
        trace_preview = q.computation_trace[:300]
        trace_md = f'\n\n<details><summary>🔢 Computation trace</summary>\n\n```\n{trace_preview}\n```\n</details>'

    display(Markdown(f"""
---
**Q{idx+1}** [{q.topic}] — _{q.difficulty}_

> {q.question}

{options_md}

💡 **Explanation:** {q.explanation[:200]}
{trace_md}
"""))

def show_flashcard(fc: Flashcard, idx: int):
    display(Markdown(f"""
---
**FC{idx+1}** [{fc.topic}]

| | |
|---|---|
| **Front** | {fc.front[:200]} |
| **Back** | {fc.back[:200]} |
"""))

def show_fillblank(fb: FillBlankQuestion, idx: int):
    blanks_str = ', '.join(f'`{b.correct_answer}`' for b in fb.blanks)
    display(Markdown(f"""
---
**FB{idx+1}** [{fb.topic}]

> {fb.template[:200]}

Answers: {blanks_str}
"""))

# Store all results
ALL_RESULTS = {}
print('Helpers loaded ✓')

Helpers loaded ✓


---
## Test 1: Accuracy — 10 Quiz Questions

**Topic:** Đạo hàm và ứng dụng  
**Checks:** Each quiz has 4 options, 1 correct, explanation, and a computation trace.

In [3]:
req1 = make_request('Đạo hàm và ứng dụng', num_questions=10, game_types=['quiz'])
res1 = await run_pipeline(req1)

out1 = res1['final_output']
assert out1 is not None, 'No output!'

stats1 = show_summary_table(out1, res1['_elapsed'], res1['iteration_count'])
ALL_RESULTS['accuracy_10q'] = stats1

2026-03-22 15:12:06 [info     ] creating_graph                


2026-03-22 15:12:06 [info     ] graph_created                 


2026-03-22 15:12:06 [info     ] graph_compiled_without_checkpointer


2026-03-22 15:12:06 [info     ] supervisor_analyzing           doc_scope=system topic='Đạo hàm và ứng dụng' user_id=accuracy_test


2026-03-22 15:12:06 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0


...

2026-03-22 15:14:46 [info     ] formatter_starting             game_types=['quiz'] items_count=10 user_id=accuracy_test


2026-03-22 15:14:56 [info     ] formatter_quiz_batch_parsed    batch_start=0 items=7


2026-03-22 15:15:04 [info     ] formatter_quiz_batch_parsed    batch_start=7 items=3


2026-03-22 15:15:04 [info     ] formatter_completed            fill_blank=0 flashcard=0 generation_time=17.370535135269165 quiz=10 total_items=10



| Metric | Value |
|--------|-------|
| Total items | **10** |
| Quiz / Flashcard / Fill-blank | 10 / 0 / 0 |
| Generated → Passed review | 10 → 10 (100%) |
| Rejected | 0 |
| Iterations | 1 |
| Time | 177.3s (17.7s per item) |


In [4]:
# Validate structure & show all questions
quizzes1 = out1.content.quiz
with_trace = [q for q in quizzes1 if q.computation_trace]

display(Markdown(f'### Computation traces: {len(with_trace)}/{len(quizzes1)}'))

for i, q in enumerate(quizzes1):
    assert len(q.options) == 4, f'Q{i+1}: Expected 4 options, got {len(q.options)}'
    assert any(o.is_correct for o in q.options), f'Q{i+1}: No correct option'
    assert q.question and q.explanation, f'Q{i+1}: Missing question or explanation'
    show_quiz_card(q, i)

display(Markdown('### ✅ Accuracy test PASSED'))

### Computation traces: 10/10


---
**Q1** [Khái niệm đạo hàm] — _DifficultyLevel.COMPREHENSION_

> Ý nghĩa của công thức định nghĩa đạo hàm của hàm số y = f(x) tại điểm x₀: f'(x₀) = lim[Δx→0] (f(x₀ + Δx) - f(x₀)) / Δx là gì?

  - ✅ Tốc độ thay đổi tức thời của hàm số f(x) tại điểm x₀.
  - ❌ Giá trị của hàm số f(x) tại điểm x₀.
  - ❌ Độ dốc trung bình của hàm số trên một khoảng.
  - ❌ Diện tích dưới đồ thị hàm số từ 0 đến x₀.

💡 **Explanation:** Công thức này biểu thị tốc độ thay đổi tức thời của hàm số f(x) tại điểm x₀. Cụ thể: Biểu thức (f(x₀ + Δx) - f(x₀)) / Δx đại diện cho tốc độ thay đổi trung bình của hàm số trên một khoảng nhỏ từ x₀ đế


<details><summary>🔢 Computation trace</summary>

```
null
```
</details>



---
**Q2** [Quy tắc tính đạo hàm cơ bản] — _DifficultyLevel.COMPREHENSION_

> Đạo hàm của một hàm hằng số f(x) = c bằng bao nhiêu?

  - ❌ 1
  - ❌ c
  - ✅ 0
  - ❌ Không xác định

💡 **Explanation:** Theo định nghĩa đạo hàm, nếu f(x) = c, thì f(x₀ + Δx) = c và f(x₀) = c. Do đó, f'(x₀) = lim[Δx→0] (c - c) / Δx = lim[Δx→0] 0 / Δx = 0. Một hàm hằng số không thay đổi giá trị, nên tốc độ thay đổi của n


<details><summary>🔢 Computation trace</summary>

```
null
```
</details>



---
**Q3** [Ứng dụng của đạo hàm (vận tốc tức thời)] — _DifficultyLevel.COMPREHENSION_

> Trong chuyển động thẳng, nếu S(t) là quãng đường đi được của một vật sau thời gian t, thì đạo hàm S'(t) biểu thị đại lượng vật lý nào?

  - ❌ Gia tốc tức thời
  - ❌ Quãng đường đi được
  - ❌ Vận tốc trung bình
  - ✅ Vận tốc tức thời

💡 **Explanation:** Đạo hàm S'(t) biểu thị vận tốc tức thời của vật tại thời điểm t. Đạo hàm của một hàm số đo lường tốc độ thay đổi tức thời của hàm số đó. S(t) là hàm quãng đường, nên S'(t) là tốc độ thay đổi của quãng


<details><summary>🔢 Computation trace</summary>

```
null
```
</details>



---
**Q4** [Quy tắc tính đạo hàm] — _DifficultyLevel.COMPREHENSION_

> Đạo hàm của hàm số f(x) = 3x² + 5x - 7 là gì?

  - ❌ f'(x) = 3x + 5
  - ❌ f'(x) = 6x - 7
  - ✅ f'(x) = 6x + 5
  - ❌ f'(x) = 3x² + 5

💡 **Explanation:** Áp dụng các quy tắc đạo hàm: (3x²)' = 6x, (5x)' = 5, (-7)' = 0. Vậy f'(x) = 6x + 5.


<details><summary>🔢 Computation trace</summary>

```
Code:
```python
import sympy
x = sympy.Symbol('x')
f_x = 3*x**2 + 5*x - 7
derivative_f_x = sympy.diff(f_x, x)
print(f'{derivative_f_x=}')
```
Output:
derivative_f_x=6*x + 5
```
</details>


...


---
**Q7** [Khái niệm đạo hàm] — _DifficultyLevel.COMPREHENSION_

> Đạo hàm của hàm số f(x) tại điểm x₀, f'(x₀), có ý nghĩa hình học gì đối với đồ thị hàm số y = f(x)?

  - ❌ Diện tích dưới đồ thị
  - ❌ Độ dài cung của đồ thị
  - ✅ Hệ số góc của tiếp tuyến
  - ❌ Tung độ của điểm trên đồ thị

💡 **Explanation:** Về mặt hình học, đạo hàm f'(x₀) chính là hệ số góc của tiếp tuyến của đồ thị hàm số y = f(x) tại điểm có hoành độ x₀. Nó cho biết độ dốc của đồ thị tại đúng điểm đó.


<details><summary>🔢 Computation trace</summary>

```
null
```
</details>



---
**Q8** [Quy tắc tính đạo hàm] — _DifficultyLevel.COMPREHENSION_

> Đạo hàm của hàm số g(x) = x⁵ - x + 10 là gì?

  - ✅ 5x⁴ - 1
  - ❌ 5x⁵ - 1
  - ❌ 5x⁴ - x + 10
  - ❌ 5x⁴

💡 **Explanation:** Áp dụng quy tắc đạo hàm của tổng/hiệu, đạo hàm của xⁿ là nxⁿ⁻¹, và đạo hàm của hằng số là 0. Do đó, g'(x) = 5x⁴ - 1.


<details><summary>🔢 Computation trace</summary>

```
Code:
    import sympy
    x = sympy.Symbol('x')
    g_x = x**5 - x + 10
    derivative_g_x = sympy.diff(g_x, x)
    print(f'{derivative_g_x=}')
    
Output:
derivative_g_x=5*x**4 - 1
```
</details>



---
**Q9** [Khái niệm đạo hàm, ứng dụng] — _DifficultyLevel.COMPREHENSION_

> Trong vật lý, đạo hàm của hàm quãng đường S(t) theo thời gian t biểu thị khái niệm nào?

  - ❌ Tốc độ trung bình
  - ✅ Tốc độ tức thời
  - ❌ Gia tốc
  - ❌ Quãng đường đi được

💡 **Explanation:** Đạo hàm của hàm quãng đường S(t) theo thời gian t chính là tốc độ tức thời của vật tại thời điểm t, mô tả tốc độ tại một khoảnh khắc cụ thể.


<details><summary>🔢 Computation trace</summary>

```
null
```
</details>



---
**Q10** [Quy tắc tính đạo hàm] — _DifficultyLevel.COMPREHENSION_

> Đạo hàm của hàm số h(x) = (x² + 1)(3x - 2) là gì?

  - ❌ 6x² - 4x
  - ✅ 9x² - 4x + 3
  - ❌ 3x² + 3
  - ❌ 2x + 3

💡 **Explanation:** Áp dụng quy tắc đạo hàm của tích (u·v)' = u'v + uv', với u = x² + 1 và v = 3x - 2. Ta có u' = 2x và v' = 3. Thay vào công thức, ta được h'(x) = (2x)(3x - 2) + (x² + 1)(3) = 6x² - 4x + 3x² + 3 = 9x² - 


<details><summary>🔢 Computation trace</summary>

```
Code:
    import sympy
    x = sympy.Symbol('x')
    h_x = (x**2 + 1) * (3*x - 2)
    derivative_h_x = sympy.diff(h_x, x)
    print(f'{derivative_h_x=}')
    
Output:
derivative_h_x=3*x**2 + 2*x*(3*x - 2) + 3
```
</details>


### ✅ Accuracy test PASSED

---
## Test 2: Vietnamese Language Quality

**Topic:** Phương trình bậc hai  
**Checks:** All question text, explanations, and flashcard fronts are in Vietnamese (≥80%).

In [5]:
req2 = make_request('Phương trình bậc hai', num_questions=10, game_types=['quiz', 'flashcard'])
res2 = await run_pipeline(req2)

out2 = res2['final_output']
assert out2 is not None, 'No output!'

stats2 = show_summary_table(out2, res2['_elapsed'], res2['iteration_count'])
ALL_RESULTS['vietnamese'] = stats2

2026-03-22 15:15:04 [info     ] creating_graph                


2026-03-22 15:15:04 [info     ] graph_created                 


2026-03-22 15:15:04 [info     ] graph_compiled_without_checkpointer


2026-03-22 15:15:04 [info     ] supervisor_analyzing           doc_scope=system topic='Phương trình bậc hai' user_id=accuracy_test


2026-03-22 15:15:04 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0


...

2026-03-22 15:17:09 [info     ] formatter_quiz_batch_parsed    batch_start=7 items=3


2026-03-22 15:17:19 [info     ] formatter_flashcard_batch_parsed batch_start=0 items=7


2026-03-22 15:17:21 [info     ] formatter_flashcard_batch_parsed batch_start=7 items=3


2026-03-22 15:17:21 [info     ] formatter_completed            fill_blank=0 flashcard=10 generation_time=27.336439847946167 quiz=10 total_items=20



| Metric | Value |
|--------|-------|
| Total items | **20** |
| Quiz / Flashcard / Fill-blank | 10 / 10 / 0 |
| Generated → Passed review | 10 → 10 (100%) |
| Rejected | 0 |
| Iterations | 1 |
| Time | 137.3s (6.9s per item) |


In [6]:
# Check Vietnamese content
vn_count, total_checked, non_vn = 0, 0, []

for q in out2.content.quiz:
    for label, text in [('Question', q.question), ('Explanation', q.explanation)]:
        total_checked += 1
        if is_vietnamese(text):
            vn_count += 1
        else:
            non_vn.append(f'{label}: {text[:80]}')

for fc in out2.content.flashcard:
    total_checked += 1
    if is_vietnamese(fc.front):
        vn_count += 1
    else:
        non_vn.append(f'FC front: {fc.front[:80]}')

vn_rate = vn_count / total_checked * 100 if total_checked > 0 else 0

display(Markdown(f'### Vietnamese content: {vn_count}/{total_checked} ({vn_rate:.0f}%)'))

if non_vn:
    display(Markdown('**⚠️ Non-Vietnamese items:**\n' + '\n'.join(f'- {x}' for x in non_vn[:5])))

# Show sample quiz & flashcard
display(Markdown('### Sample Quiz'))
if out2.content.quiz:
    show_quiz_card(out2.content.quiz[0], 0)

display(Markdown('### Sample Flashcard'))
if out2.content.flashcard:
    show_flashcard(out2.content.flashcard[0], 0)

assert vn_rate >= 80, f'Expected >=80% Vietnamese, got {vn_rate:.0f}%'
display(Markdown('### ✅ Vietnamese test PASSED'))

### Vietnamese content: 30/30 (100%)

### Sample Quiz


---
**Q1** [Định nghĩa phương trình bậc hai] — _DifficultyLevel.COMPREHENSION_

> Theo định nghĩa, điều kiện nào sau đây xác định một phương trình bậc hai có dạng `ax² + bx + c = 0`?

  - ❌ a = 0
  - ❌ b ≠ 0
  - ✅ a ≠ 0
  - ❌ c = 0

💡 **Explanation:** Phương trình bậc hai tổng quát có dạng `ax² + bx + c = 0` với điều kiện `a ≠ 0`.


<details><summary>🔢 Computation trace</summary>

```
null
```
</details>


### Sample Flashcard


---
**FC1** [Định nghĩa phương trình bậc hai]

| | |
|---|---|
| **Front** | Dựa vào định nghĩa, hãy giải thích vì sao phương trình `3x² - 5x + 2 = 0` là một phương trình bậc hai, và phương trình `2x - 7 = 0` không phải là phương trình bậc hai. |
| **Back** | Phương trình `3x² - 5x + 2 = 0` là phương trình bậc hai vì có dạng `ax² + bx + c = 0` với `a = 3 ≠ 0`. Phương trình `2x - 7 = 0` không phải là phương trình bậc hai vì hệ số `a = 0`. Phương trình bậc h |


### ✅ Vietnamese test PASSED

---
## Test 3: Content Diversity — 20 Questions

**Topic:** Hàm số và đồ thị  
**Checks:** All questions are unique (≥90%), multiple sub-topics.

In [7]:
req3 = make_request('Hàm số và đồ thị', num_questions=20, game_types=['quiz'])
res3 = await run_pipeline(req3)

out3 = res3['final_output']
assert out3 is not None, 'No output!'

stats3 = show_summary_table(out3, res3['_elapsed'], res3['iteration_count'])
ALL_RESULTS['diversity_20q'] = stats3

2026-03-22 15:17:21 [info     ] creating_graph                


2026-03-22 15:17:21 [info     ] graph_created                 


2026-03-22 15:17:21 [info     ] graph_compiled_without_checkpointer


2026-03-22 15:17:21 [info     ] supervisor_analyzing           doc_scope=system topic='Hàm số và đồ thị' user_id=accuracy_test


2026-03-22 15:17:21 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0


...

2026-03-22 15:19:15 [info     ] formatter_quiz_batch_parsed    batch_start=0 items=7


2026-03-22 15:19:25 [info     ] formatter_quiz_batch_parsed    batch_start=7 items=7


2026-03-22 15:19:37 [info     ] formatter_quiz_batch_parsed    batch_start=14 items=6


2026-03-22 15:19:37 [info     ] formatter_completed            fill_blank=0 flashcard=0 generation_time=32.24025797843933 quiz=20 total_items=20



| Metric | Value |
|--------|-------|
| Total items | **20** |
| Quiz / Flashcard / Fill-blank | 20 / 0 / 0 |
| Generated → Passed review | 20 → 20 (100%) |
| Rejected | 0 |
| Iterations | 1 |
| Time | 136.2s (6.8s per item) |


In [8]:
quizzes3 = out3.content.quiz
questions3 = [q.question for q in quizzes3]
unique3 = set(questions3)
dup_rate = (len(questions3) - len(unique3)) / len(questions3) * 100 if questions3 else 0

display(Markdown(f'### Unique questions: {len(unique3)}/{len(questions3)} (dup rate: {dup_rate:.0f}%)'))

# Sub-topic distribution
topics3 = {}
for q in quizzes3:
    topics3[q.topic] = topics3.get(q.topic, 0) + 1

topic_rows = '\n'.join(f'| {t} | {c} |' for t, c in sorted(topics3.items(), key=lambda x: -x[1]))
display(Markdown(f"""
### Sub-topic Distribution ({len(topics3)} topics)

| Topic | Count |
|-------|-------|
{topic_rows}
"""))

assert len(unique3) >= len(questions3) * 0.9, f'Too many duplicates: {len(unique3)}/{len(questions3)}'
assert len(topics3) >= 2, f'Expected >=2 sub-topics, got {len(topics3)}'
display(Markdown('### ✅ Diversity test PASSED'))

### Unique questions: 20/20 (dup rate: 0%)


### Sub-topic Distribution (16 topics)

| Topic | Count |
|-------|-------|
| Cực trị hàm số | 3 |
| Hàm số bậc hai, đỉnh parabol | 2 |
| Đạo hàm | 2 |
| Khái niệm hàm số | 1 |
| Tập xác định của hàm số | 1 |
| Đồ thị hàm số | 1 |
| Hàm số đồng biến, nghịch biến | 1 |
| Hàm số chẵn, lẻ | 1 |
| Hàm số bậc nhất | 1 |
| Hàm số bậc nhất, đồng biến/nghịch biến | 1 |
| Hàm số bậc hai, đồ thị parabol | 1 |
| Hàm số bậc hai, trục đối xứng | 1 |
| Đạo hàm, đồng biến/nghịch biến | 1 |
| Đạo hàm cấp hai | 1 |
| Hàm số mũ | 1 |
| Hàm số logarit | 1 |


### ✅ Diversity test PASSED

---
## Test 4: Scale Test — 30 Questions (Kahoot Scale)

**Topic:** Tích phân và ứng dụng  
**Checks:** Pipeline handles 30Q (≥70% yield), batch parsing works at scale.

In [9]:
req4 = make_request('Tích phân và ứng dụng', num_questions=30, game_types=['quiz'])
res4 = await run_pipeline(req4)

out4 = res4['final_output']
assert out4 is not None, f'No output. Errors: {res4.get("errors", [])}'

stats4 = show_summary_table(out4, res4['_elapsed'], res4['iteration_count'])
ALL_RESULTS['scale_30q'] = stats4

2026-03-22 15:19:37 [info     ] creating_graph                


2026-03-22 15:19:37 [info     ] graph_created                 


2026-03-22 15:19:37 [info     ] graph_compiled_without_checkpointer


2026-03-22 15:19:37 [info     ] supervisor_analyzing           doc_scope=system topic='Tích phân và ứng dụng' user_id=accuracy_test


2026-03-22 15:19:37 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0


...

2026-03-22 15:24:00 [info     ] formatter_quiz_batch_parsed    batch_start=7 items=7


2026-03-22 15:24:12 [info     ] formatter_quiz_batch_parsed    batch_start=14 items=7


2026-03-22 15:24:20 [info     ] formatter_quiz_batch_parsed    batch_start=21 items=4


2026-03-22 15:24:20 [info     ] formatter_completed            fill_blank=0 flashcard=0 generation_time=38.177329301834106 quiz=25 total_items=25



| Metric | Value |
|--------|-------|
| Total items | **25** |
| Quiz / Flashcard / Fill-blank | 25 / 0 / 0 |
| Generated → Passed review | 39 → 25 (64%) |
| Rejected | 14 |
| Iterations | 1 |
| Time | 282.7s (11.3s per item) |


In [10]:
quizzes4 = out4.content.quiz
n4 = len(quizzes4)
assert n4 >= 21, f'Expected >=21 (70% of 30), got {n4}'

unique4 = set(q.question for q in quizzes4)
display(Markdown(f'### Unique: {len(unique4)}/{n4}'))

# Show first, middle, last
display(Markdown('### Sample Items (first / middle / last)'))
for idx in [0, n4 // 2, n4 - 1]:
    show_quiz_card(quizzes4[idx], idx)

display(Markdown(f'### ✅ Scale test PASSED — {n4} items in {res4["_elapsed"]:.0f}s ({res4["_elapsed"]/n4:.1f}s/item)'))

### Unique: 25/25

### Sample Items (first / middle / last)


---
**Q1** [Ứng dụng của tích phân trong vật lý] — _DifficultyLevel.COMPREHENSION_

> Một vật chuyển động với vận tốc v(t) = 20 - 40t (m/s). Tính quãng đường vật đi được trong 0.5 giây đầu tiên.

  - ✅ 5m
  - ❌ 10m
  - ❌ 0m
  - ❌ -5m

💡 **Explanation:** Quãng đường vật đi được trong 0.5 giây đầu tiên là tích phân của hàm vận tốc v(t) từ 0 đến 0.5. Kết quả là 5m.


<details><summary>🔢 Computation trace</summary>

```
Code:
import sympy
t = sympy.Symbol('t')
v_t = 20 - 40*t
integral_result = sympy.integrate(v_t, (t, 0, 0.5))
print(f'{integral_result=}')
Output:
integral_result=5.00000000000000
```
</details>



---
**Q13** [Tích phân hàm mũ] — _DifficultyLevel.COMPREHENSION_

> Tính tích phân xác định: $\int_{0}^{1} e^{2x} dx$

  - ✅ $\frac{e^2 - 1}{2}$
  - ❌ $e^2 - 1$
  - ❌ $2e^2 - 2$
  - ❌ $\frac{e^2 + 1}{2}$

💡 **Explanation:** Tích phân của $e^{2x}$ là $\frac{1}{2}e^{2x}$. Thay cận từ 0 đến 1, ta có $\frac{1}{2}e^{2 \cdot 1} - \frac{1}{2}e^{2 \cdot 0} = \frac{1}{2}e^2 - \frac{1}{2}e^0 = \frac{1}{2}e^2 - \frac{1}{2} = \frac{


<details><summary>🔢 Computation trace</summary>

```
Code:
import sympy
x = sympy.Symbol('x')
f_x = sympy.exp(2*x)
integral_result = sympy.integrate(f_x, (x, 0, 1))
print(f'{integral_result=}')
Output:
integral_result=-1/2 + exp(2)/2
```
</details>



---
**Q25** [Tích phân xác định] — _DifficultyLevel.COMPREHENSION_

> Giá trị của tích phân $\int_{0}^{2} (x^2 - x) dx$ là:

  - ✅ $\frac{2}{3}$
  - ❌ $\frac{4}{3}$
  - ❌ $-\frac{4}{3}$
  - ❌ 0

💡 **Explanation:** Nguyên hàm của $x^2 - x$ là $\frac{x^3}{3} - \frac{x^2}{2}$. Vậy $[\frac{x^3}{3} - \frac{x^2}{2}]_{0}^{2} = (\frac{2^3}{3} - \frac{2^2}{2}) - 0 = \frac{8}{3} - 2 = \frac{2}{3}$.


<details><summary>🔢 Computation trace</summary>

```
Code:
import sympy
x = sympy.Symbol('x')
f_x = x**2 - x
integral_result = sympy.integrate(f_x, (x, 0, 2))
print(f'{integral_result=}')
Output:
integral_result=2/3
```
</details>


### ✅ Scale test PASSED — 25 items in 283s (11.3s/item)

---
## Test 5: All Game Types — Quiz + Flashcard + Fill-blank

**Topic:** Chuyển động thẳng đều và biến đổi đều (20Q)  
**Checks:** All 3 game formats are generated and structurally valid.

In [11]:
req5 = make_request(
    'Chuyển động thẳng đều và biến đổi đều',
    num_questions=20,
    game_types=['quiz', 'flashcard', 'fill_blank'],
)
res5 = await run_pipeline(req5)

out5 = res5['final_output']
assert out5 is not None, 'No output!'

stats5 = show_summary_table(out5, res5['_elapsed'], res5['iteration_count'])
ALL_RESULTS['all_types'] = stats5

2026-03-22 15:24:20 [info     ] creating_graph                


2026-03-22 15:24:20 [info     ] graph_created                 


2026-03-22 15:24:20 [info     ] graph_compiled_without_checkpointer


2026-03-22 15:24:20 [info     ] supervisor_analyzing           doc_scope=system topic='Chuyển động thẳng đều và biến đổi đều' user_id=accuracy_test


2026-03-22 15:24:20 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0


...

2026-03-22 15:26:55 [info     ] formatter_fill_blank_batch_parsed batch_start=0 items=7


2026-03-22 15:27:01 [info     ] formatter_fill_blank_batch_parsed batch_start=7 items=7


2026-03-22 15:27:07 [info     ] formatter_fill_blank_batch_parsed batch_start=14 items=6


2026-03-22 15:27:07 [info     ] formatter_completed            fill_blank=20 flashcard=20 generation_time=63.25637626647949 quiz=20 total_items=60



| Metric | Value |
|--------|-------|
| Total items | **60** |
| Quiz / Flashcard / Fill-blank | 20 / 20 / 20 |
| Generated → Passed review | 20 → 20 (100%) |
| Rejected | 0 |
| Iterations | 1 |
| Time | 166.7s (2.8s per item) |


In [12]:
qn5, fn5, bn5 = len(out5.content.quiz), len(out5.content.flashcard), len(out5.content.fill_blank)
assert qn5 >= 10, f'Quiz: expected >=10, got {qn5}'
assert fn5 >= 10, f'Flashcard: expected >=10, got {fn5}'
assert bn5 >= 10, f'Fill-blank: expected >=10, got {bn5}'

display(Markdown('### Sample Quiz'))
show_quiz_card(out5.content.quiz[0], 0)

display(Markdown('### Sample Flashcard'))
show_flashcard(out5.content.flashcard[0], 0)

display(Markdown('### Sample Fill-blank'))
show_fillblank(out5.content.fill_blank[0], 0)

display(Markdown('### ✅ All game types test PASSED'))

### Sample Quiz


---
**Q1** [Chuyển động thẳng đều] — _DifficultyLevel.COMPREHENSION_

> Đặc điểm nào sau đây mô tả đúng về vận tốc của vật trong chuyển động thẳng đều?

  - ❌ Vận tốc thay đổi về độ lớn nhưng không đổi về hướng.
  - ✅ Vận tốc không đổi cả về độ lớn và hướng.
  - ❌ Vận tốc không đổi về độ lớn nhưng thay đổi về hướng.
  - ❌ Vận tốc luôn bằng 0.

💡 **Explanation:** Trong chuyển động thẳng đều, vận tốc của vật là một đại lượng không đổi cả về độ lớn và hướng.


<details><summary>🔢 Computation trace</summary>

```
null
```
</details>


### Sample Flashcard


---
**FC1** [Chuyển động thẳng đều]

| | |
|---|---|
| **Front** | Trong chuyển động thẳng đều, vận tốc của vật có những đặc điểm gì? |
| **Back** | Trong chuyển động thẳng đều, vận tốc của vật là một đại lượng không đổi cả về độ lớn và hướng. Chuyển động thẳng đều được định nghĩa là chuyển động có quỹ đạo là đường thẳng và vận tốc tức thời không  |


### Sample Fill-blank


---
**FB1** [Chuyển động thẳng đều]

> Trong chuyển động thẳng đều, vận tốc của vật là một đại lượng ___ cả về ___ và ___.

Answers: `không đổi`, `độ lớn`, `hướng`


### ✅ All game types test PASSED

---
## Final Summary

In [13]:
# Build summary table
rows = []
total_items = 0
total_time = 0.0
pass_rates = []

for name, s in ALL_RESULTS.items():
    if s is None:
        rows.append(f'| {name} | ❌ FAILED | | | | | |')
        continue
    pr = s['passed'] / s['generated'] * 100 if s['generated'] > 0 else 0
    pass_rates.append(pr)
    total_items += s['total']
    total_time += s['elapsed']
    rows.append(f'| {name} | {s["total"]} | {s["quiz"]} | {s["fc"]} | {s["fb"]} | {pr:.0f}% | {s["elapsed"]:.0f}s |')

avg_pr = sum(pass_rates) / len(pass_rates) if pass_rates else 0
rows.append(f'| **TOTAL** | **{total_items}** | | | | **{avg_pr:.0f}%** | **{total_time:.0f}s** |')

table = '\n'.join(rows)

failed = [k for k, v in ALL_RESULTS.items() if v is None]
status = '❌ Failed: ' + ', '.join(failed) if failed else '✅ All tests passed'

display(Markdown(f"""
### Results

| Test | Items | Quiz | FC | FB | Pass Rate | Time |
|------|-------|------|----|----|-----------|------|
{table}

**Tests:** {len(ALL_RESULTS)} | **Items:** {total_items} | **Time:** {total_time:.0f}s ({total_time/60:.1f} min)

### {status}
"""))


### Results

| Test | Items | Quiz | FC | FB | Pass Rate | Time |
|------|-------|------|----|----|-----------|------|
| accuracy_10q | 10 | 10 | 0 | 0 | 100% | 177s |
| vietnamese | 20 | 10 | 10 | 0 | 100% | 137s |
| diversity_20q | 20 | 20 | 0 | 0 | 100% | 136s |
| scale_30q | 25 | 25 | 0 | 0 | 64% | 283s |
| all_types | 60 | 20 | 20 | 20 | 100% | 167s |
| **TOTAL** | **135** | | | | **93%** | **900s** |

**Tests:** 5 | **Items:** 135 | **Time:** 900s (15.0 min)

### ✅ All tests passed
